# ToysAD-8K Official Frozen Inference

**Notebook file:** `10a1-toysad8k-official-unseen-frozen-inference.ipynb`

This notebook runs the final frozen automatic route on the official model-facing RGB resources. It verifies the exact Notebook 08F method, Notebook 09B review policy, and completed Notebook 09D protocol receipt before opening official RGB. LocateAnything proposes boxes, SAM 2.1 produces masks, DINOv2-small extracts features, label-free matching connects objects across views, and P1 fusion selects the odd-one-out candidate.

Seen-test and unseen-test predictions remain separate. This entry point is permanently pinned to the `unseen` cohort. `VPI_OFFICIAL_COHORT` may repeat that value, but any conflicting value stops the run before inputs are opened. The notebook never mounts or opens evaluator resources and reports no accuracy metrics.

## Execution and restart contract

- One shard is completed for each selected cohort before a longer run.
- `gate`, `incremental`, and `full` phases are explicit.
- Incremental runs process a bounded number of new shards and preserve exact receipts.
- A later Kaggle version attaches exactly one private recovery output and restores it with `VPI_RESUME_ROOT`.
- A shard is reused only when its complete pipeline identity, input hash, output hash, and embedding hashes match.
- Failures and insufficient-evidence cases are retained as results.
- The selected prompt and pinned model configuration must rebuild the exact Notebook 08F base-model identity before GPU inference starts.
- Method, policy, and internal decision prerequisites come from exact minimal hash-locked datasets. Complete evaluator notebook outputs are forbidden mounts.
- Prediction panels show boxes, an all-object SAM overlay, the selected binary mask, the selected masked-on-black object, identities, P1 ranks, fused odd-one-out, margin, agreement, and review signal. They contain no truth colouring.
- Gate and incremental runs export a hash-locked recovery ZIP, record an aggregate ClearML recovery task, and remove duplicate working state only after the ZIP passes its CRC check.
- Only a full per-split prediction lock and the final two-split summary carry the `final` ClearML tag.

In [ ]:
from __future__ import annotations

from hashlib import sha256
import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image as NotebookImage, display

sys.dont_write_bytecode = True
NOTEBOOK_FILE = "10a1-toysad8k-official-unseen-frozen-inference.ipynb"
RUN_PHASE = os.getenv("VPI_RUN_PHASE", "gate").strip().lower()
PINNED_OFFICIAL_COHORT = "unseen"
REQUESTED_OFFICIAL_COHORT = os.getenv("VPI_OFFICIAL_COHORT", "unseen").strip().lower()
if RUN_PHASE not in {"gate", "incremental", "full"}:
    raise ValueError("VPI_RUN_PHASE must be gate, incremental, or full")
if REQUESTED_OFFICIAL_COHORT != PINNED_OFFICIAL_COHORT:
    raise ValueError(
        f"{NOTEBOOK_FILE} is pinned to {PINNED_OFFICIAL_COHORT}; "
        f"VPI_OFFICIAL_COHORT requested {REQUESTED_OFFICIAL_COHORT!r}"
    )
OFFICIAL_COHORT = PINNED_OFFICIAL_COHORT
EXECUTION_ENVIRONMENT = "kaggle" if Path("/kaggle/input").is_dir() else "local"
OUTPUT_ROOT = Path(os.getenv(
    "VPI_OUTPUT_ROOT",
    "/kaggle/working/toysad8k_official_inference" if EXECUTION_ENVIRONMENT == "kaggle" else ".local/runs/toysad8k_official_inference",
)).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
def kaggle_roots() -> list[Path]:
    root = Path("/kaggle/input")
    return sorted((path for path in root.iterdir() if path.is_dir()), key=lambda path: path.as_posix()) if root.is_dir() else []


def find_code_root() -> Path:
    configured = os.getenv("VPI_CODE_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend((Path.cwd(), *Path.cwd().parents, *kaggle_roots()))
    for candidate in candidates:
        if (candidate / "scripts/run_official_inference.py").is_file():
            return candidate.resolve()
        if (candidate / "code/scripts/run_official_inference.py").is_file():
            return (candidate / "code").resolve()
    raise FileNotFoundError("Could not locate the Visual Peer Inspector code snapshot")


def find_named_file(name: str, environment_name: str) -> Path:
    configured = os.getenv(environment_name)
    if configured and Path(configured).expanduser().is_file():
        return Path(configured).expanduser().resolve()
    matches = sorted({path.resolve() for root in kaggle_roots() for path in root.rglob(name) if path.is_file()})
    if not matches:
        raise FileNotFoundError(f"Could not locate {name}")
    digests = {sha256(path.read_bytes()).hexdigest() for path in matches}
    if len(digests) != 1:
        raise ValueError(f"Mounted copies of {name} differ")
    return matches[0]


def find_release_recovery_pair(role: str) -> tuple[Path, Path] | None:
    configured = os.getenv("VPI_RESUME_ROOT")
    sources = [Path(configured).expanduser()] if configured else kaggle_roots()
    bundles = sorted({
        path.resolve()
        for source in sources
        for path in source.rglob(f"official-{role}-release-recovery.zip")
        if path.is_file()
    })
    manifests = sorted({
        path.resolve()
        for source in sources
        for path in source.rglob(f"official-{role}-release-recovery-manifest.json")
        if path.is_file()
    })
    if not bundles and not manifests:
        return None
    if len(bundles) != 1 or len(manifests) != 1:
        raise ValueError("VPI_RESUME_ROOT must contain exactly one official release recovery pair")
    return bundles[0], manifests[0]


def code_environment() -> dict[str, str]:
    environment = os.environ.copy()
    source_root = str(CODE_ROOT / "src")
    existing = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = source_root if not existing else source_root + os.pathsep + existing
    return environment


def read_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)


def enable_tracking(environment: dict[str, str]) -> str:
    access = read_secret("CLEARML_API_ACCESS_KEY")
    secret = read_secret("CLEARML_API_SECRET_KEY")
    if not access or not secret:
        return "disabled"
    environment.update({
        "CLEARML_API_ACCESS_KEY": access,
        "CLEARML_API_SECRET_KEY": secret,
        "CLEARML_API_HOST": "https://api.clear.ml",
        "CLEARML_WEB_HOST": "https://app.clear.ml",
        "CLEARML_FILES_HOST": "https://files.clear.ml",
    })
    return "online"


def prepare_tracking_runtime(mode: str, environment: dict[str, str]) -> str:
    if mode != "online":
        return mode
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements/confirmation-tracking-lock.txt")],
            check=True,
            timeout=180,
            env=environment,
        )
    except Exception as error:
        print(f"ClearML dependency setup was unavailable: {type(error).__name__}")
        return "disabled"
    return mode

In [ ]:
def find_model_input(cohort_id: str) -> Path:
    for root in kaggle_roots():
        for candidate in (root, *(path for path in root.iterdir() if path.is_dir())):
            record_path = candidate / "package_record.json"
            if not record_path.is_file():
                continue
            try:
                record = json.loads(record_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if record.get("cohort_id") == cohort_id and (candidate / "media").is_dir():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate model-facing package {cohort_id}")


def find_recovery_pair(cohort_id: str) -> tuple[Path, Path] | None:
    configured = os.getenv("VPI_RESUME_ROOT")
    if not configured:
        return None
    source = Path(configured).expanduser().resolve(strict=True)
    if any(term in source.name.lower() for term in ("evaluator", "label", "crosswalk", "ground-truth")):
        raise ValueError("A model run cannot restore evaluator content")
    bundles = sorted(source.rglob(f"{cohort_id}-recovery.zip"))
    manifests = sorted(source.rglob(f"{cohort_id}-recovery-manifest.json"))
    if len(bundles) != 1 or len(manifests) != 1:
        raise ValueError("VPI_RESUME_ROOT must contain exactly one recovery ZIP and its manifest")
    return bundles[0], manifests[0]

## Verify the frozen prerequisites before RGB access

In [ ]:
CODE_ROOT = find_code_root()
sys.path.insert(0, str(CODE_ROOT / "src"))
from visual_peer_inspector.notebook_provenance import verify_notebook_code_snapshot

CODE_PROVENANCE = verify_notebook_code_snapshot(
    CODE_ROOT,
    "notebooks/official-evaluation/" + NOTEBOOK_FILE,
)

METHOD_FREEZE = find_named_file("method_freeze.json", "VPI_METHOD_FREEZE")
APPLICATION_CONTRACT = find_named_file("application_output_contract.json", "VPI_APPLICATION_CONTRACT")
POLICY_FREEZE = find_named_file("review_policy_freeze.json", "VPI_POLICY_FREEZE")
LOCK_RESULT_RECEIPT = find_named_file("internal_lock_result_receipt.json", "VPI_LOCK_RESULT_RECEIPT")

from visual_peer_inspector.workflows.official_inference import load_lock_result_receipt
from visual_peer_inspector.workflows.protected_cohort_inference import load_method_freeze, load_policy_freeze

method_record = load_method_freeze(
    METHOD_FREEZE,
    application_contract_path=APPLICATION_CONTRACT,
    require_resource=True,
)
policy_record = load_policy_freeze(
    POLICY_FREEZE,
    method_sha256=method_record["sha256"],
    require_resource=True,
)
lock_result = load_lock_result_receipt(
    LOCK_RESULT_RECEIPT,
    method_sha256=method_record["sha256"],
    policy_sha256=policy_record["sha256"],
    require_resource=True,
)
display({
    "method SHA-256": method_record["sha256"],
    "policy SHA-256": policy_record["sha256"],
    "internal-lock result": lock_result["status"],
    "view contracts": method_record["view_contracts"],
})

## Locate model inputs and prepare the pinned GPU runtime

In [ ]:
environment = code_environment()
SEEN_INPUT = UNSEEN_INPUT = None
cohort_id = "toysad8k-official-seen-test-715-v1" if OFFICIAL_COHORT == "seen" else "toysad8k-official-unseen-test-2437-v1"
selected_input = find_model_input(cohort_id)
if OFFICIAL_COHORT == "seen":
    SEEN_INPUT = selected_input
else:
    UNSEEN_INPUT = selected_input
environment["SAM2_BUILD_CUDA"] = "0"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements/confirmation-lock.txt")],
    check=True, timeout=900, env=environment,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4"],
    check=True, timeout=600, env=environment,
)
import torch
from huggingface_hub import hf_hub_download
from visual_peer_inspector.data.model_safe_package import file_sha256
from visual_peer_inspector.models.sam2 import Sam2Configuration
if not torch.cuda.is_available():
    raise RuntimeError("Official inference requires a Kaggle GPU")
sam_configuration = Sam2Configuration()
SAM_CHECKPOINT = Path(hf_hub_download(
    repo_id=sam_configuration.model_repository,
    filename=sam_configuration.checkpoint_filename,
    revision=sam_configuration.model_revision,
    cache_dir="/tmp/vpi-sam21-model-cache",
))
if SAM_CHECKPOINT.stat().st_size != sam_configuration.checkpoint_size_bytes or file_sha256(SAM_CHECKPOINT) != sam_configuration.checkpoint_sha256:
    raise RuntimeError("The pinned SAM 2.1 checkpoint identity differs")
SELECTED_PROMPT = find_named_file("selected_prompt.json", "VPI_SELECTED_PROMPT")
display({"GPU": torch.cuda.get_device_name(0), "GPU count": torch.cuda.device_count(), "cohort": OFFICIAL_COHORT})

## Run the selected restartable phase

In [ ]:
maximum_new_shards = int(os.getenv("VPI_MAXIMUM_NEW_SHARDS", "10"))
tracking_mode = prepare_tracking_runtime(enable_tracking(environment), environment)
command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_official_inference.py"),
    "--cohort", OFFICIAL_COHORT,
    "--run-mode", RUN_PHASE,
    "--method-freeze", str(METHOD_FREEZE),
    "--application-contract", str(APPLICATION_CONTRACT),
    "--policy-freeze", str(POLICY_FREEZE),
    "--lock-result-receipt", str(LOCK_RESULT_RECEIPT),
    "--output-root", str(OUTPUT_ROOT),
    "--device", "cuda",
    "--shard-size", "100",
    "--maximum-new-shards", str(maximum_new_shards),
    "--clearml-mode", tracking_mode,
]
if SEEN_INPUT is not None:
    command.extend(["--seen-model-input", str(SEEN_INPUT)])
if UNSEEN_INPUT is not None:
    command.extend(["--unseen-model-input", str(UNSEEN_INPUT)])
if SELECTED_PROMPT is not None and SAM_CHECKPOINT is not None:
    command.extend(["--selected-prompt", str(SELECTED_PROMPT), "--sam-checkpoint", str(SAM_CHECKPOINT)])
cohort_id = "toysad8k-official-seen-test-715-v1" if OFFICIAL_COHORT == "seen" else "toysad8k-official-unseen-test-2437-v1"
recovery = find_recovery_pair(cohort_id)
if recovery is not None:
    command.extend(["--recovery-bundle", str(recovery[0]), "--recovery-manifest", str(recovery[1])])
if RUN_PHASE in {"gate", "incremental"}:
    command.extend(["--export-recovery", "--prune-after-recovery"])
for mounted in kaggle_roots():
    command.extend(["--mounted-root", str(mounted)])
preflight_path = OUTPUT_ROOT / f"preflight/{OFFICIAL_COHORT}-model-preflight.json"
preflight_command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_frozen_development_pipeline.py"),
    "preflight",
    "--cohort", f"official-{OFFICIAL_COHORT}",
    "--model-input-root", str(selected_input),
    "--selected-prompt", str(SELECTED_PROMPT),
    "--sam-checkpoint", str(SAM_CHECKPOINT),
    "--output-path", str(preflight_path),
    "--device", "cuda",
    "--maximum-image-attempts", "5",
]
subprocess.run(preflight_command, check=True, timeout=1800, env=environment)
preflight_record = json.loads(preflight_path.read_text(encoding="utf-8"))
if preflight_record["status"] != "passed":
    raise RuntimeError("The bounded model preflight did not pass")
environment["HF_HUB_OFFLINE"] = "1"
environment["TRANSFORMERS_OFFLINE"] = "1"
subprocess.run(command, check=True, env=environment)
display({"phase": RUN_PHASE, "cohort": OFFICIAL_COHORT, "model preflight": preflight_record["status"], "maximum new shards per stage": maximum_new_shards})

## Inspect prediction-only evidence

In [ ]:
for cohort_id in (
    "toysad8k-official-seen-test-715-v1",
    "toysad8k-official-unseen-test-2437-v1",
):
    root = OUTPUT_ROOT / cohort_id
    progress_path = root / "incremental_progress.json"
    if progress_path.is_file():
        display(pd.DataFrame(json.loads(progress_path.read_text(encoding="utf-8"))["stages"]).T)
    safe_root = root / "safe"
    inventory_path = root / "review-only/qualitative_evidence_inventory.json"
    if inventory_path.is_file():
        display(pd.read_csv(safe_root / "view_contract_summary.csv"))
        display(pd.read_csv(safe_root / "review_signal_summary.csv"))
        inventory = json.loads(inventory_path.read_text(encoding="utf-8"))
        for item in inventory["items"][:3]:
            display(NotebookImage(filename=str(root / item["prediction_panel"])))
            display(NotebookImage(filename=str(root / item["sam_evidence_panel"])))

publish_root = OUTPUT_ROOT / "publish" / f"official-{OFFICIAL_COHORT}-prediction"
if publish_root.is_dir():
    display({"private prediction resource": str(publish_root)})

## Publication and output contract

After a full seen run, publish only `publish/official-seen-prediction` as the private dataset `vpi-toysad-official-seen-prediction`. After a full unseen run, publish only `publish/official-unseen-prediction` as `vpi-toysad-official-unseen-prediction`. Each resource contains its locked model-only artifacts and sanitized run record. Do not publish recovery state as a final prediction resource. Notebook 10A Finalize attaches both prediction resources and builds one minimal cross-split summary without opening evaluator data.